# Stage 6 — Build the graph and run the core analysis

The analytical heart. Build the directed predicate graph (edge points predicate → citing device),
confirm it is acyclic, then compute the **persistent-recalled-predicate** analysis and all
sensitivity subsets.

Because this corpus (154 codes) differs from the original 9-code study, the headline numbers are
**computed here, not assumed**. This notebook writes `data/expected_values.json` — the analyst's
authoritative results, which Stage 8 (verify) reproduces independently.

Network: none. Fully deterministic — re-run anytime, it always reproduces.

In [ ]:
import json
import pandas as pd
import networkx as nx

corpus = pd.read_csv("data/corpus.csv", dtype=str)
corpus["decision_year"] = pd.to_numeric(corpus["decision_year"], errors="coerce")
edges = pd.read_csv("data/predicate_edges.csv", dtype=str)
recalled = pd.read_csv("data/recalled_nodes.csv", dtype=str)

year = dict(zip(corpus["k_number"], corpus["decision_year"]))
recall_date = dict(zip(recalled["k_number"], pd.to_datetime(recalled["event_date_initiated"], errors="coerce")))
recall_class = dict(zip(recalled["k_number"], recalled["worst_class"]))
recalled_set = set(recalled["k_number"]) & set(corpus["k_number"])

### Build the DiGraph (all corpus devices are nodes)

In [ ]:
G = nx.DiGraph()
G.add_nodes_from(corpus["k_number"])
for _, r in edges.iterrows():
    if r["predicate_knumber"] in G and r["device_knumber"] in G:
        G.add_edge(r["predicate_knumber"], r["device_knumber"], confidence=r["confidence"])

is_dag = nx.is_directed_acyclic_graph(G)
connected = [n for n in G if G.degree(n) > 0]
depth = nx.dag_longest_path_length(nx.condensation(G))  # Option A: longest predicate chain via condensation; handles the lone mutual-citation 2-cycle without altering the edge set
largest_wcc = max((len(c) for c in nx.weakly_connected_components(G)), default=0)
print(f"CHECKPOINT  nodes {G.number_of_nodes()} | edges {G.number_of_edges()} | DAG {is_dag}")
print(f"CHECKPOINT  connected {len(connected)} | largest component {largest_wcc} | max depth {depth}")

### Recall prevalence

In [ ]:
recalled_nodes = recalled_set & set(G.nodes())
pct = round(100 * len(recalled_nodes) / G.number_of_nodes(), 1)
print(f"CHECKPOINT  recalled nodes {len(recalled_nodes)} = {pct}% of all nodes")

### Persistent recalled predicates

A recalled device A is a *persistent recalled predicate* if a later device B was cleared **after**
A's recall date and B is itself **not** recalled.

In [ ]:
def analysis(edge_df, recalled_ok):
    """edge_df: edges to use; recalled_ok: set of recalled predicates to consider."""
    H = nx.DiGraph()
    H.add_nodes_from(corpus["k_number"])
    for _, r in edge_df.iterrows():
        if r["predicate_knumber"] in H and r["device_knumber"] in H:
            H.add_edge(r["predicate_knumber"], r["device_knumber"])
    preds, cits, downstream = set(), 0, set()
    for A in recalled_ok:
        if A not in H:
            continue
        rd = recall_date.get(A)
        if pd.isna(rd):
            continue
        for B in H.successors(A):
            if B in recalled_set:
                continue
            by = year.get(B)
            if pd.notna(by) and by > rd.year:
                preds.add(A); cits += 1; downstream.add(B)
    return len(preds), cits, len(downstream)

base = analysis(edges, recalled_nodes)
print(f"CHECKPOINT  persistent predicates {base[0]} | citations {base[1]} | downstream {base[2]}")

# first cited only AFTER own recall
first_after = 0
for A in recalled_nodes:
    rd = recall_date.get(A)
    if A not in G or pd.isna(rd):
        continue
    citer_years = [year.get(B) for B in G.successors(A) if pd.notna(year.get(B))]
    if citer_years and min(citer_years) > rd.year:
        first_after += 1
print(f"CHECKPOINT  first cited AFTER own recall {first_after}")

# max recall-to-latest-citation gap
max_gap = 0.0
for A in recalled_nodes:
    rd = recall_date.get(A)
    if A not in G or pd.isna(rd):
        continue
    cyrs = [year.get(B) for B in G.successors(A) if pd.notna(year.get(B))]
    if cyrs:
        max_gap = max(max_gap, max(cyrs) - rd.year)
print(f"CHECKPOINT  max recall-to-latest gap {max_gap:.1f} yr")

### Sensitivity subsets

In [ ]:
# Class I/II only
cii = {A for A in recalled_nodes if recall_class.get(A) in ("Class I", "Class II")}
s_cii = analysis(edges, cii)
# high-confidence edges only
hc_edges = edges[edges["confidence"] == "SECTION_HEADED"]
s_hc = analysis(hc_edges, recalled_nodes)
# combined
s_comb = analysis(hc_edges, cii)
print(f"CHECKPOINT  class_I_II {s_cii} | high_conf {s_hc} | combined {s_comb}")

### Write the authoritative expected values (for Stage 8)

In [ ]:
cov = pd.read_csv("data/coverage_diagnostic.csv")
covc = cov["coverage"].value_counts(); ncov = len(cov)
EXPECTED = {
    "snapshot_date": json.load(open("snapshot/SNAPSHOT.json"))["snapshot_date"],
    "corpus_devices": int(len(corpus)),
    "edges": int(G.number_of_edges()),
    "is_dag": bool(is_dag),
    "connected_nodes": int(len(connected)),
    "max_chain_depth": int(depth),
    "largest_component": int(largest_wcc),
    "recalled_nodes": int(len(recalled_nodes)),
    "pct_recalled": pct,
    "persistent_predicates": base[0], "post_recall_citations": base[1], "downstream_devices": base[2],
    "first_after_recall": first_after,
    "max_gap_years": round(max_gap, 1),
    "sens_class_I_II": list(s_cii),
    "sens_high_conf": list(s_hc),
    "sens_combined": list(s_comb),
    "cov_resolvable_pct": round(100*covc.get("edge_resolvable",0)/ncov, 1),
    "cov_name_only_pct": round(100*covc.get("name_only",0)/ncov, 1),
    "cov_out_of_scope_pct": round(100*covc.get("out_of_scope",0)/ncov, 1),
}
json.dump(EXPECTED, open("data/expected_values.json", "w"), indent=2)
nx.write_gml(G, "data/predicate_graph.gml")
print("CHECKPOINT  wrote data/expected_values.json and data/predicate_graph.gml")
print(json.dumps(EXPECTED, indent=2))